In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

def minmax_normalize(df):
    return pd.DataFrame(MinMaxScaler().fit_transform(df), columns=df.columns, index=df.index)

def compute_nmf_pearson(real_df, synth_df, n_components=1, title="NMF Pearson Heatmap"):
    # Transpose if samples are columns (fix shape)
    real_df = real_df.T
    synth_df = synth_df.T

    # Normalize
    real_df = minmax_normalize(real_df)
    synth_df = minmax_normalize(synth_df)

    # Align columns
    synth_df.columns = real_df.columns
    common_cols = real_df.columns.intersection(synth_df.columns)
    real_df = real_df[common_cols]
    synth_df = synth_df[common_cols]

    # Apply NMF
    nmf_real = NMF(n_components=n_components, init='random', random_state=0)
    nmf_synth = NMF(n_components=n_components, init='random', random_state=0)

    W_real = nmf_real.fit_transform(real_df)
    W_synth = nmf_synth.fit_transform(synth_df)

    # Ensure same number of rows
    min_rows = min(W_real.shape[0], W_synth.shape[0])
    W_real = W_real[:min_rows, :]
    W_synth = W_synth[:min_rows, :]

    # Compute Pearson correlation matrix
    corr_matrix = np.zeros((W_real.shape[1], W_synth.shape[1]))
    for i in range(W_real.shape[1]):
        for j in range(W_synth.shape[1]):
            corr_matrix[i, j], _ = stats.pearsonr(W_real[:, i], W_synth[:, j])

    # Print average Pearson correlation
    avg_corr = np.nanmean(corr_matrix)
    print(f"\n📊 {title}")
    print(f"Average Pearson Correlation: {avg_corr:.4f}")




# ---- Load Fixed Data and Run Evaluation ----

# Read the CSV assuming first row contains proper headers
real_expr = pd.read_csv("scAEGAN/labelled_latent_meth.csv", header=0)
fake_expr = pd.read_csv("scAEGAN/outdataA.csv", header=0)

# Drop the first row of actual data if it was originally part of column names
real_expr = real_expr.iloc[1:].reset_index(drop=True)
fake_expr = fake_expr.iloc[1:].reset_index(drop=True)

# Convert to numeric (safety step in case types are object due to header row)
real_expr = real_expr.apply(pd.to_numeric)
fake_expr = fake_expr.apply(pd.to_numeric)

# Now compute the metrics
expr_corr = compute_nmf_pearson(real_expr, fake_expr, title="Expression NMF Pearson Correlation")



📊 Expression NMF Pearson Correlation
Average Pearson Correlation: 0.9126


In [8]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.metrics import pairwise_distances
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


def minmax_normalize(df):
    return pd.DataFrame(MinMaxScaler().fit_transform(df), columns=df.columns, index=df.index)

def kde_jsd(p, q, num_points=1000):
    # Kernel density estimation
    kde_p = gaussian_kde(p)
    kde_q = gaussian_kde(q)

    xs = np.linspace(min(p.min(), q.min()), max(p.max(), q.max()), num_points)
    p_pdf = kde_p(xs)
    q_pdf = kde_q(xs)

    # Normalize (KDE may not sum to 1)
    p_pdf /= p_pdf.sum()
    q_pdf /= q_pdf.sum()

    return jensenshannon(p_pdf, q_pdf) ** 2  # squared JSD (optional)

def histogram_intersection(p, q, bins=50):
    hist_p, _ = np.histogram(p, bins=bins, density=True)
    hist_q, _ = np.histogram(q, bins=bins, density=True)

    # Normalize manually to sum to 1
    hist_p = hist_p / np.sum(hist_p)
    hist_q = hist_q / np.sum(hist_q)

    return np.sum(np.minimum(hist_p, hist_q))


def compute_nmf_and_metrics(real_df, synth_df, n_components=1, title="NMF Evaluation"):
    # Transpose if needed
    real_df = real_df
    synth_df = synth_df

    # Normalize
    real_df = minmax_normalize(real_df)
    synth_df = minmax_normalize(synth_df)

    # Align columns
    synth_df.columns = real_df.columns
    common_cols = real_df.columns.intersection(synth_df.columns)
    real_df = real_df[common_cols]
    synth_df = synth_df[common_cols]

    # Apply NMF
    nmf_real = NMF(n_components=n_components, init='random', random_state=0)
    nmf_synth = NMF(n_components=n_components, init='random', random_state=0)

    W_real = nmf_real.fit_transform(real_df)
    W_synth = nmf_synth.fit_transform(synth_df)

    # Clip to same sample size
    min_rows = min(W_real.shape[0], W_synth.shape[0])
    W_real = W_real[:min_rows, :]
    W_synth = W_synth[:min_rows, :]

    # Initialize metric holders
    pearson_corrs = []
    wasserstein_dists = []
    hist_inters = []
    jsds = []
    ks_stats = []

    # For each component, compare real vs synthetic
    for i in range(W_real.shape[1]):
        x = W_real[:, i]
        y = W_synth[:, i]

        # Pearson
        corr, _ = stats.pearsonr(x, y)
        pearson_corrs.append(corr)

        # Wasserstein
        wasserstein_dists.append(wasserstein_distance(x, y))

        # Histogram Intersection
        hist_inters.append(histogram_intersection(x, y))

        # JSD via KDE
        jsds.append(kde_jsd(x, y))

        # KS
        ks_stats.append(ks_2samp(x, y).statistic)

    # Summary
    print(f"\n📊 {title}")
    print(f"🔹 Avg Pearson Correlation:       {np.nanmean(pearson_corrs):.4f}")
    print(f"🔹 Avg Wasserstein Distance:     {np.mean(wasserstein_dists):.4f}")
    print(f"🔹 Avg Histogram Intersection:   {np.mean(hist_inters):.4f}")
    print(f"🔹 Avg JSD (KDE):                {np.mean(jsds):.4f}")
    print(f"🔹 Avg KS Statistic:             {np.mean(ks_stats):.4f}")


# ---- Load Data ----
real_expr = pd.read_csv("scAEGAN/latent_meth.csv", header=0)
fake_expr = pd.read_csv("scAEGAN/outdataB.csv", header=0)

# Keep first column as index (sample IDs)
real_expr.set_index(real_expr.columns[0], inplace=True)
fake_expr.set_index(fake_expr.columns[0], inplace=True)

# For real_expr, drop the last column (condition or label column)
real_expr = real_expr.iloc[:, :-1]
fake_expr = fake_expr.iloc[:, :-1]

# Convert only the numeric part
real_expr = real_expr.apply(pd.to_numeric, errors='coerce')
fake_expr = fake_expr.apply(pd.to_numeric, errors='coerce')

# Run evaluation
compute_nmf_and_metrics(real_expr, fake_expr, n_components=1, title="Expression NMF Evaluation")



📊 Expression NMF Evaluation
🔹 Avg Pearson Correlation:       0.9924
🔹 Avg Wasserstein Distance:     0.0062
🔹 Avg Histogram Intersection:   0.8454
🔹 Avg JSD (KDE):                0.0165
🔹 Avg KS Statistic:             0.1445


In [12]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance, ks_2samp
from scipy.stats import gaussian_kde


def minmax_normalize(df):
    return pd.DataFrame(MinMaxScaler().fit_transform(df), columns=df.columns, index=df.index)

def kde_jsd(p, q, num_points=1000):
    kde_p = gaussian_kde(p)
    kde_q = gaussian_kde(q)
    xs = np.linspace(min(p.min(), q.min()), max(p.max(), q.max()), num_points)
    p_pdf = kde_p(xs); q_pdf = kde_q(xs)
    p_pdf /= p_pdf.sum(); q_pdf /= q_pdf.sum()
    return jensenshannon(p_pdf, q_pdf) ** 2

def histogram_intersection(p, q, bins=50):
    hist_p, _ = np.histogram(p, bins=bins, density=True)
    hist_q, _ = np.histogram(q, bins=bins, density=True)
    hist_p = hist_p / np.sum(hist_p); hist_q = hist_q / np.sum(hist_q)
    return np.sum(np.minimum(hist_p, hist_q))

def compute_nmf_and_metrics(real_df, synth_df, n_components=1, title="NMF Evaluation"):
    # Align columns (features)
    common_cols = real_df.columns.intersection(synth_df.columns)
    real_df = real_df[common_cols]
    synth_df = synth_df[common_cols]

    # Normalize each matrix separately
    real_df = minmax_normalize(real_df)
    synth_df = minmax_normalize(synth_df)

    # Apply NMF
    nmf_real = NMF(n_components=n_components, init='random', random_state=0)
    nmf_synth = NMF(n_components=n_components, init='random', random_state=0)
    W_real = nmf_real.fit_transform(real_df)
    W_synth = nmf_synth.fit_transform(synth_df)

    # Clip to same sample size
    m = min(W_real.shape[0], W_synth.shape[0])
    W_real = W_real[:m, :]
    W_synth = W_synth[:m, :]

    pearson_corrs, wasserstein_dists, hist_inters, jsds, ks_stats = [], [], [], [], []

    for i in range(W_real.shape[1]):
        x = W_real[:, i]; y = W_synth[:, i]
        corr, _ = stats.pearsonr(x, y)
        pearson_corrs.append(corr)
        wasserstein_dists.append(wasserstein_distance(x, y))
        hist_inters.append(histogram_intersection(x, y))
        jsds.append(kde_jsd(x, y))
        ks_stats.append(ks_2samp(x, y).statistic)

    print(f"\n📊 {title}")
    print(f"  Real n={W_real.shape[0]} | Synth n={W_synth.shape[0]}")
    print(f"  Avg Pearson Correlation:     {np.nanmean(pearson_corrs):.4f}")
    print(f"  Avg Wasserstein Distance:    {np.mean(wasserstein_dists):.4f}")
    print(f"  Avg Histogram Intersection:  {np.mean(hist_inters):.4f}")
    print(f"  Avg JSD (KDE):               {np.mean(jsds):.4f}")
    print(f"  Avg KS Statistic:            {np.mean(ks_stats):.4f}")

# ---- Load Data ----
real_raw = pd.read_csv("scAEGAN/latent_meth_OS.csv", header=0)
fake_raw = pd.read_csv("scAEGAN/outdataB_OS.csv", header=0)

# Keep first column as index (sample IDs)
real_raw.set_index(real_raw.columns[0], inplace=True)
fake_raw.set_index(fake_raw.columns[0], inplace=True)

# Extract condition columns (assumed to be the LAST column in each file)
real_cond = pd.to_numeric(real_raw.iloc[:, -1], errors='coerce')
fake_cond = pd.to_numeric(fake_raw.iloc[:, -1], errors='coerce')

# Feature matrices (drop the last column which is condition/label)
real_feat = real_raw.iloc[:, :-1].apply(pd.to_numeric, errors='coerce')
fake_feat = fake_raw.iloc[:, :-1].apply(pd.to_numeric, errors='coerce')

# Optional: drop columns that are entirely NaN after coercion
real_feat = real_feat.dropna(axis=1, how='all')
fake_feat = fake_feat.dropna(axis=1, how='all')

# ---- Evaluate by condition ----
for cond in [0, 1]:
    real_sub = real_feat.loc[real_cond == cond]
    fake_sub = fake_feat.loc[fake_cond == cond]

    if real_sub.empty or fake_sub.empty:
        print(f"\n⚠️ Skipping condition {cond}: "
              f"real_sub empty={real_sub.empty}, fake_sub empty={fake_sub.empty}")
        continue

    compute_nmf_and_metrics(
        real_sub,
        fake_sub,
        n_components=1,
        title=f"Expression NMF Evaluation (condition={cond})"
    )



📊 Expression NMF Evaluation (condition=0)
  Real n=324 | Synth n=324
  Avg Pearson Correlation:     0.9921
  Avg Wasserstein Distance:    0.0017
  Avg Histogram Intersection:  0.8611
  Avg JSD (KDE):               0.0015
  Avg KS Statistic:            0.0525

📊 Expression NMF Evaluation (condition=1)
  Real n=324 | Synth n=324
  Avg Pearson Correlation:     0.9899
  Avg Wasserstein Distance:    0.0008
  Avg Histogram Intersection:  0.8796
  Avg JSD (KDE):               0.0002
  Avg KS Statistic:            0.0370
